In [1]:
%matplotlib QtAgg
import numpy as np
import pandas as pd
import random as random
import copy
import matplotlib.pyplot as plt


class Capa:
    w : np.ndarray
    y: np.ndarray
    delta: np.ndarray

    def __init__(self, w_i, y_i, delta_i):
        self.w = w_i
        self.y = y_i
        self.delta = delta_i

    def mostrar(self):
        print(f"Pesos: {self.w}")
        print(f"Salidas: {self.y}")
        print(f"Deltas: {self.delta}\n")


def sigm(x):
    return (2/(1+np.exp(-x))) - 1

entrada_usuario = [8,1]

tabla = pd.read_csv('../../Data/gtp_2/concent_trn.csv', header=None).to_numpy()
x0 = -np.ones(len(tabla))
entradas = np.c_[x0, tabla[:,:-1]] # indice -1 := ultima columna ( Acceso a indices con : es [) )
yd =  tabla[:, -1]

print(entradas.shape)
print(yd.shape)

w = np.random.rand(entrada_usuario[0], len(entradas[0])) - 0.5 # dimension: 0 == columnas, dimension: 1 == filas
y_init = np.zeros(entrada_usuario[0])
delta = np.zeros(entrada_usuario[0])
cap = Capa(w,y_init,delta)
vect_capas = [copy.deepcopy(cap)]

# Iniciar red (aleatorio):
for i in range(1,len(entrada_usuario)):
    w = np.random.rand(entrada_usuario[i], entrada_usuario[i-1]+1) - 0.5
    y_init = np.zeros(entrada_usuario[i])
    delta = np.zeros(entrada_usuario[i])
    cap = Capa(w,y_init,delta)
    vect_capas.append(copy.deepcopy(cap))


# Visualizar red inicial:
print(f"Cantidad de capas: {len(vect_capas)}\n")
print(f"Red neuronal: \n")
i = 1
for capa in vect_capas:
    print(f"Capa: {i}")
    capa.mostrar()
    i += 1


(1499, 3)
(1499,)
Cantidad de capas: 2

Red neuronal: 

Capa: 1
Pesos: [[-0.37627349 -0.01690391  0.1930875 ]
 [ 0.18579554 -0.34576978  0.05279385]
 [-0.2245942   0.04625845  0.0217245 ]
 [-0.19164757  0.03976661 -0.28057856]
 [-0.18977444  0.02610452  0.05666285]
 [-0.00331921 -0.18158576  0.42658318]
 [-0.22816432 -0.47784675 -0.41724466]
 [-0.47912859 -0.34362806 -0.13856624]]
Salidas: [0. 0. 0. 0. 0. 0. 0. 0.]
Deltas: [0. 0. 0. 0. 0. 0. 0. 0.]

Capa: 2
Pesos: [[-0.2385727   0.41315208 -0.29116718 -0.33844109 -0.10890573  0.33001243
  -0.44324096 -0.15092391  0.04297006]]
Salidas: [0.]
Deltas: [0.]



In [2]:
# Iterar sobre la red:

epoca = 1
epoca_max = 1000
mu = 0.8


# arr_ptos_positivos = np.ndarray()
# arr_ptos_negativos = np.ndarray()

# n -> ejemplo actual
# i -> la capa
# j -> la neurona
while epoca < epoca_max: 
    # SOLUCIÓN: Iterar sobre len(entradas) (las filas), no len(entradas[0]) (las columnas)
    for n in range(len(entradas)):

        # paso hacia adelante
        for i in range(len(vect_capas)):
            for j in range(len(vect_capas[i].y)):
                if i==0:
                    z = np.dot(entradas[n,:],vect_capas[i].w[j,:])
                else: 
                    ent = np.r_[-1,vect_capas[i-1].y]
                    z = np.dot(ent,vect_capas[i].w[j,:])
                vect_capas[i].y[j] = sigm(z)

        # propagacion hacia atras
        # SOLUCIÓN: Empezar en len(vect_capas)-1 para no tener un IndexError
        for i in range(len(vect_capas)-1,-1,-1):
            for j in range(len(vect_capas[i].y)):
                # SOLUCIÓN: Restar 1 al if porque los índices empiezan en 0
                if i==len(vect_capas)-1:
                    # SOLUCIÓN: Usar yd[n] (el deseado del ejemplo actual) en vez de yd[j]
                    vect_capas[i].delta[j] = (1/2) * (yd[n] - vect_capas[i].y[j]) * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])
                else:
                    # SOLUCIÓN: np.dot contra el array de deltas completo de la capa siguiente (quitamos el [j])
                    vect_capas[i].delta[j] = (1/2) * np.dot(vect_capas[i+1].delta, vect_capas[i+1].w[:,j+1])  * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])

        #actualizar los pesos
        for i in range(len(vect_capas)):
            for j in range(len(vect_capas[i].y)):
                # SOLUCIÓN: Usar len de los pesos de la neurona actual
                for m in range(len(vect_capas[i].w[j])):
                    if i==0:
                        # SOLUCIÓN: Usar += en lugar de -=
                        vect_capas[i].w[j,m] += mu*vect_capas[i].delta[j]*entradas[n,m]
                    else:
                        # SOLUCIÓN: Hay que reconstruir la entrada con el -1 para poder usar el índice 'm' sin salir de rango, y usar +=
                        ent = np.r_[-1, vect_capas[i-1].y]
                        vect_capas[i].w[j,m] += mu*vect_capas[i].delta[j]*ent[m]
                        
    # Verificación:
    acierto = 0

    for n in range(len(entradas)):
        for capa in range(len(vect_capas)):
            
            for neuron in range(len(vect_capas[capa].y)):
                if capa==0:
                    z = np.dot(entradas[n,:],vect_capas[capa].w[neuron,:])
                else:                
                    ent = np.r_[-1,vect_capas[capa-1].y]
                    z = np.dot(ent,vect_capas[capa].w[neuron,:])
                
                vect_capas[capa].y[neuron] = sigm(z)


        salida_red = vect_capas[-1].y[-1]

        if ((salida_red > 0.0 and yd[n] == 1) or (salida_red < 0.0 and yd[n] == -1)):
            acierto += 1

    tasa_acierto = acierto/len(entradas)
    # print(f"Fin entrenamiento. Epoca: {epoca}, Tasa de acierto: {tasa_acierto * 100: .2f}\n")
    print(f"{tasa_acierto}")

    if tasa_acierto > 0.80 and mu == 0.8:
        mu = 0.5
    elif tasa_acierto >= 0.92 and mu == 0.5:
        mu = 0.3
    elif tasa_acierto >= 0.96 and mu == 0.3:
        mu = 0.1
    elif tasa_acierto >= 0.98 and mu == 0.1:
        mu = 0.05
    elif tasa_acierto >= 0.9889:
        print(f"Convergencia. Epoca {epoca}, Tasa aciertos: {tasa_acierto}")
        break

    # if (tasa_acierto>=0.8 and mu==0.8):
    #     mu = 0.5
    # if (tasa_acierto>=0.92 and mu==0.5):
    #     mu = 0.3
    # if (tasa_acierto>=0.96 and mu==0.3):
    #     mu = 0.1
    # if (tasa_acierto>=0.98 and mu==0.1):
    #     mu = 0.05
    
    epoca += 1





0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.4589726484322882
0.5196797865243495
0.5323549032688459
0.543028685790527
0.7004669779853235
0.7118078719146097
0.7158105403602402
0.72515010006

### Test